# Imports

In [45]:
import os
import sys
import cv2 #For blur detection
import json
import math
import time
import datetime
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from pathml.core import HESlide 
from pathml.preprocessing import TissueDetectionHE, LabelWhiteSpaceHE, LabelArtifactTileHE, StainNormalizationHE
from matplotlib import pyplot as plt
prj = 'HCC-CBS-175-Hillman-RBao-panCancer-HE'
base = Path('/ix/rbao/Projects/%s' % prj)
data = base.joinpath('data','acral_melanoma_sarah')
print(data.exists())
results = base.joinpath('results','infer_outputs','tile_dfs')
problem_file = 'S08-4387.svs'
df = pd.read_csv(results.joinpath('infer_%s_tiles_df.tsv' % problem_file.split('.')[0]),
                 sep='\t')
fn = data.joinpath(problem_file)
print(fn.exists(),df.shape)
df.head()

True
True (31170, 7)


,tile,x,y,sz,cur_path,p_pos,pred_cls
0,S08-4387_n5677_x73696_y3136_px224.jpg,73696.0,3136.0,224.0,/scratch/slurm-2189695/results/tiles/S08-4387/...,0.001319,1
1,S08-4387_n5678_x73920_y3136_px224.jpg,73920.0,3136.0,224.0,/scratch/slurm-2189695/results/tiles/S08-4387/...,0.000214,1
2,S08-4387_n5679_x74144_y3136_px224.jpg,74144.0,3136.0,224.0,/scratch/slurm-2189695/results/tiles/S08-4387/...,0.003203,1
3,S08-4387_n5680_x74368_y3136_px224.jpg,74368.0,3136.0,224.0,/scratch/slurm-2189695/results/tiles/S08-4387/...,0.011879,1
4,S08-4387_n5681_x74592_y3136_px224.jpg,74592.0,3136.0,224.0,/scratch/slurm-2189695/results/tiles/S08-4387/...,0.000828,1


In [41]:
wsi = HESlide(str(fn))
tissue_detect = TissueDetectionHE(mask_name='tissue',
                                  outer_contours_only= True,
                                  blur_ksize=21,
                                  threshold = 20) #Thresh possibly off for smaller tile size (<=224 )?

blank_detect=LabelWhiteSpaceHE(label_name='blank',
                               proportion_threshold=0.9, #Thresh too low?
                               )

art_detect = LabelArtifactTileHE(label_name='artifact')
normalizer = StainNormalizationHE(target = "normalize",
                                  stain_estimation_method = "macenko")


# TILE WSI AND FILTER TILES BASED ON VARIOUS CRITERIA:
tiles = wsi.generate_tiles(shape=tile_size,
                           stride=tile_stride,
                           pad=False,
                           level=0)

		Beginning to preprocess file (not distributed).


In [62]:
def collect_issues(skips_df,tile,i,iii,problem,value):
    y,x = tile.coords #Flipped to match qupath x,y in v9
    skips_df.loc[iii,'n'] = i
    skips_df.loc[iii,'problem']=problem
    skips_df.loc[iii,'value'] = value
    skips_df.loc[iii,'x'] = x
    skips_df.loc[iii,'y'] = y
    # im = np.array(tile.image).reshape(-1)
    # start = 5
    # stop = len(im) + start
    # skips_df.iloc[iii,start:stop] = im
    iii = iii + 1
    return skips_df, iii

In [65]:
n = df.tile.str.split('_').str[1].str.split('n').str[1].astype(int)
ii=0
tile_df = pd.DataFrame([])
skips_df = pd.DataFrame([])
iii = 0
bgr = (2,1,0)
tiles = wsi.generate_tiles(shape=tile_size,
                           stride=tile_stride,
                           pad=False,
                           level=0)
tile_size = 224
tile_stride = tile_size # tile_size//2
tissue_thresh = 70 # % of tile that contains tissue
gj_anno_overlap = 10 # If assigning tiles to geojson this percent of tile must be inside annotation to be included
blur_thresh = 40 #threshold for blurriness calculated by strength of edges in image (higher = sharper image)
use_stain_norm = True
tot = 127 * tile_size**2
tot_tiles_est=(wsi.shape[0]//tile_size) * (wsi.shape[1]//tile_size) * (2*(tile_size//tile_stride))
t_stack=np.array([])
for i,tile in enumerate(tqdm(tiles)):
    if (not np.any(n == i)): #If tile was never saved in first pass, examine it again
        blank_detect.apply(tile)    
        if tile.labels['blank'] == False:
            art_detect.apply(tile)
            #In addition -- detect pen / glass edge artifacts??
            if tile.labels['artifact'] == False:           
                im = np.array(tile.image) # r g b page order
                blur_detect = cv2.Laplacian(im[:,:,bgr], cv2.CV_64F).var()
                if (blur_detect > blur_thresh):
                    tissue_detect.apply(tile)
                    tissue_mask=tile.masks['tissue']
                    per = 100 * np.sum(tissue_mask.flatten())/tot
                    if (per > tissue_thresh) :
                        if use_stain_norm:
                            normalizer.apply(tile)
                            im = np.array(tile.image) # r g b page order
                        y,x = tile.coords #Flipped to match qupath x,y in v9
                        tile_fn = '%s_n%d_x%d_y%d_px%d.jpg' % (slide_num,i,x,y,tile_size)
                        tile_df.loc[ii,'tile'] = tile_fn
                        tile_df.loc[ii,'x'] = x
                        tile_df.loc[ii,'y'] = y
                        tile_df.loc[ii,'sz'] = tile_size
                        print('\t\t\t--%d) Saving tile %s (%d/%d)' % 
                              (ii,tile_dest.joinpath(tile_fn),i,tot_tiles_est))

                        img_fn = tile_dest.joinpath(tile_fn)
                        data = Image.fromarray(im.astype(np.uint8))
                        data.save(img_fn, quality=95)

                        #Potentially: if geojson associated with wsi, determine if tile overlaps with annotations
                        

                        # cur_time = time.time()
                        # dur = str(datetime.timedelta(seconds=(cur_time-start)))
                        print('\t\t\t\t %3.1f%% complete. %s H:M:S' % (i/tot_tiles_est*100, dur))
                        ii+=1 #Counter of included tiles (excludes blanks/artifacts/blur etc)
                    else:
                        skips_df, iii = collect_issues(skips_df,tile, i,iii, 'percent_tissue',per)
                else:
                    skips_df, iii = collect_issues(skips_df,tile, i, iii, 'blur', blur_detect)
            else:
                skips_df, iii = collect_issues(skips_df,tile, i, iii, 'artifact',tile.labels['artifact'])
        else:
            skips_df, iii = collect_issues(skips_df,tile, i, iii, 'blank',tile.labels['blank'])


121094it [18:45, 107.57it/s]


In [67]:
newfn= results.parent.joinpath('S08-4387_skipped_tiles_list.tsv')
skips_df.to_csv(newfn,sep = '\t')

In [70]:
skips_df.groupby('problem')['n'].count()

problem
artifact          17620
blank             71802
blur                 51
percent_tissue      451
Name: n, dtype: int64

# Examine 2 regions known to have tissue but tiles getting dropped:

In [73]:
idx = (skips_df.x.values > 32732) & (skips_df.x.values < 35443) & (skips_df.y.values > 43611) & (skips_df.y.values < 46069)
print(np.sum(idx))
skips_df.loc[idx,:].groupby('problem')['n'].count()

81


problem
artifact    81
Name: n, dtype: int64

In [74]:
idx = (skips_df.x.values > 34401) & (skips_df.x.values < 45620) & (skips_df.y.values > 30476) & (skips_df.y.values < 39516)
print(np.sum(idx))
skips_df.loc[idx,:].groupby('problem')['n'].count()

1421


problem
artifact          1418
percent_tissue       3
Name: n, dtype: int64

# Conclusion: they are getting scored incorrectly as artifacts

In [82]:
ignore_artifacts = True #True
artifact = True
if (ignore_artifacts or artifact == False):
    print('Ignore / treat as no artifact')
elif artifact == True:
    print('Artifact found')


Ignore / treat as no artifact
